# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U openai==2.36.0 "openai[realtime]" sounddevice==0.5.5 dotenv==0.9.9 azure-ai-voicelive==1.2.0 websocket-client==1.9.0 aiohttp==3.14.0

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_URI = os.getenv("AZURE_OPENAI_URI")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_REALTIME_DEPLOYMENT = os.getenv("AZURE_OPENAI_REALTIME_DEPLOYMENT")
AZURE_SPEECH_LEXICON_URL = os.getenv("AZURE_SPEECH_LEXICON_URL") # 각자의 lexicon.xml URL을 .env 파일에 설정해주세요.

# Microsoft Foundry Voice models
Microsoft foundry 는 다양한 voice models 을 제공한다. Microsoft 에서 직접 개발하는 MAI 모델이나 OpenAI 의 gpt 모델 등이 있는데, 사용자 어플리케이션은 최적화되어 있는 모델들이 필요하고 그 목적도 다양하다. Text-to-speech, speech-to-text 그리고 realtime 모델 등에 걸쳐 다양한 voice models 을 Microsoft foundry 에서 만나보자.

## Azure Speech Voice Live vs gpt-realtime

`gpt-realtime` 과 `Azure Speech Voice Live` 는 같은 음성-음성 모델 사용이 가능하지만, 구현 레벨에서 다음이 다르다.

| 구분 | gpt-realtime | Azure Speech Voice Live |
|------|----------------|----------------|
| **연결/SDK** | OpenAI Realtime API (low-level) / `openai`  | Azure Speech Voice Live API (high-level) / `azure-ai-voicelive` |
| **파이프라인** | speech-to-speech (S2S) | S2S: gpt-realtime models <br> Cascaded(STT → LLM → TTS): OpenAI models |
| **보이스** | OpenAI native voice 지원 | S2S: OpenAI native voice 지원 <br> Cascaded: Azure Speech Voice (Neural, DragonHD) 지원 |
| **도구 호출(RAG)** | 지원 | 지원 |
| **음성향상** | 클라이언트측 Audio Echo Cancelation (AEC) 사용 필수 | 서버측 AEC, Audio Noise Reduction (ANR) 지원 |
| **끼어들기(Barge-in)** | 클라이언트측 AEC 사용하에 구현 | 서버측 AEC, ANR 을 활용하여 구현 |
| **입·출력 커스터마이징** | 불가 | Phrase list (인식 가중) 및 Custom Lexicon (발음 교정) 지원 |
 
- <small> `gpt-realtime` 은 API 기능으로서 barge-in은 지원하지만, 서버 AEC가 없어 스피커 환경에선 AI 음성이 마이크로 되먹어 오작동한다. 따라서 gpt-realtime에서 끼어들기를 제대로 쓰려면 헤드폰/에코제거 하드웨어/클라이언트 AEC 가 필요하다.</small>

## Voice live

Azure Microsoft Foundry Voice Live API 는 gpt-realtime 같은 음성-음성(speech-to-speech) 모델을 하나의 WebSocket 엔드포인트로 감싸고, 여기에 서버측 음질 향상 (에코 제거·딥 노이즈 억제), Azure 뉴럴 보이스, (선택) 아바타·표정(viseme) 까지 얹어주는 통합 음성 대화 서비스다. 앞의 gpt-realtime 셀을 직접 WebSocket 으로 다룬 것과 달리, `azure-ai-voicelive` SDK 의 `connect()` 한 번으로 동일한 음성 챗봇을 더 간단하게 만들 수 있다.

### 동작 원리 (블록도)

```text
   🎙️ 마이크 (PCM16 24kHz)
        │  input_audio_buffer.append (base64)
        ▼
  ╔═ Azure Microsoft Foundry · Voice Live API (WebSocket) ════════
  ║   ① 입력 전처리   : 에코 제거 + 딥 노이즈 억제 (서버측)
  ║   ② Server VAD    : 발화 시작/종료 감지 → 턴 분리
  ║   ③ gpt-realtime  : 음성을 직접 이해·추론·생성 (speech-to-speech)
  ║   ④ 출력          : Azure 뉴럴 보이스 TTS + 전사(transcript)
  ╚═══════════════════════════════════════════════════════════════
        │  response.audio.delta (PCM bytes)
        ▼
   🔊 스피커 (PCM16 24kHz)
```

**위 흐름이 `run_voicelive` 코드의 이벤트에 그대로 대응한다.**

| 단계 | Voice Live 이벤트 / 처리 |
|------|--------------------------|
| <small>마이크 → 서버 전송</small> | <small>`input_audio_buffer.append`</small> |
| <small>입력 전처리(AEC·노이즈)</small> | <small>`input_audio_echo_cancellation` · `input_audio_noise_reduction`</small> |
| <small>사용자 발화 시작</small> | <small>`ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED`</small> |
| <small>끼어들기</small> | <small>`speaker.clear()` + `conn.response.cancel()`</small> |
| <small>응답 생성 시작</small> | <small>`ServerEventType.RESPONSE_CREATED`</small> |
| <small>응답 오디오 수신</small> | <small>`ServerEventType.RESPONSE_AUDIO_DELTA`</small> |
| <small>응답 텍스트 수신</small> | <small>`ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DELTA`</small> |
| <small>사용자 음성 인식 결과</small> | <small>`ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED`</small> |
| <small>도구 호출 인자 완성</small> | <small>`ServerEventType.RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE` <br> → `on_tool()`<br> → `FunctionCallOutputItem` + `response.create()`</small> |
| <small>응답 종료</small> | <small>`ServerEventType.RESPONSE_DONE`</small> |

In [ ]:
import json
import asyncio
import base64
import threading
import array

import sounddevice as sd
from azure.core.credentials import AzureKeyCredential
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import (
    RequestSession, Modality, InputAudioFormat, OutputAudioFormat,
    AzureSemanticVadMultilingual, ServerEventType, AzureStandardVoice, AudioInputTranscriptionOptions,
    AudioEchoCancellation, AudioNoiseReduction, FunctionTool, ToolChoiceLiteral,
    FunctionCallOutputItem,
)

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 스피커 — 페이드아웃 + 세대(generation) 차단
#  clear() 가 세대를 올려, 끊긴 응답의 늦게 도착한 오디오를 add() 가 전부 버린다(잔여음 누수 0)
class Speaker:
    def __init__(self, samplerate=24000, blocksize=1200, fade_sec=0.5):   # 1200 frames = 50ms
        self.buf, self.pos = bytearray(), 0
        self.lock = threading.Lock()
        self.gen = 0                              # 응답 세대
        self.fade = int(samplerate * fade_sec)    # 끼어들기 페이드아웃 길이(샘플)
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=blocksize, callback=self._cb)

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            avail = len(self.buf) - self.pos
            n = min(need, avail)
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n]); self.pos += n
            if n < need:
                outdata[n:] = b"\x00" * (need - n)   # 부족분은 무음
            if self.pos > 2_000_000:
                del self.buf[:self.pos]; self.pos = 0

    def add(self, pcm, gen):
        with self.lock:
            if gen != self.gen:        # 끊긴(옛 세대) 응답 오디오는 버림 → "다른 음성" 누수 차단
                return
            self.buf.extend(pcm)

    def clear(self):                   # 끼어들기: 세대 증가 + 페이드아웃
        with self.lock:
            self.gen += 1              # 이후 도착하는 옛 응답 delta 전부 무시
            cur = bytes(self.buf[self.pos:]); self.buf = bytearray(); self.pos = 0
            s = array.array("h"); s.frombytes(cur[:self.fade * 2]); m = len(s)
            for i in range(m):
                s[i] = int(s[i] * (m - i) / m)   # 선형 페이드아웃(즉시 끊으면 '탁' 클릭음)
            self.buf.extend(s.tobytes())

    def start(self):
        self.stream.start()
    def close(self):
        try: self.stream.stop()
        finally: self.stream.close()


def _build_voice(voice, lexicon_url=None):
    """보이스 자동 분기: '-'/':' 포함이면 Azure 뉴럴 보이스, 아니면 OpenAI 네이티브 보이스(문자열)."""
    if not isinstance(voice, str):
        return voice                       # 이미 구성된 voice 객체는 그대로
    if ("-" in voice) or (":" in voice):   # 예: ko-kr-Hyunsu:DragonHDLatestNeural
        kw = {"name": voice}
        if lexicon_url:
            kw["custom_lexicon_url"] = lexicon_url   # 발음 사전(Azure 보이스 전용)
        return AzureStandardVoice(**kw)
    return voice                           # 예: marin, alloy (OpenAI 보이스)


async def run_voicelive(
    model: str,
    voice,
    instructions: str,
    *,
    lexicon_url: str | None = None,        # 출력 TTS 발음 사전 (Azure 보이스 전용)
    phrase_list: list | None = None,       # 입력 STT 힌트 (cascaded/azure-speech 전용)
    transcription_model: str = "azure-speech",
    language: str = "ko",
    tools: list | None = None,             # RAG 등 function tool 목록
    on_tool=None,                          # async (name, args)->str : 도구 실행 콜백
    vad_threshold: float = 0.5,
    silence_ms: int = 500,
    fade_sec: float = 0.5,                 # 끼어들기 페이드아웃 길이(초)
    intro: str = "🎙️  말해보세요.\n",
):
    """Voice Live 음성 대화 공용 함수.
    - cascaded : model=gpt-5.4(chat) + voice=DragonHD(Azure)  → lexicon/phrase_list 사용 가능
    - S2S      : model=gpt-realtime  + voice=OpenAI 보이스(marin 등)
    서버 AEC·끼어들기(response.cancel)·세대 차단(끊긴 응답 잔여음 제거)·페이드아웃 공통 적용.
    """
    reset_portaudio()
    speaker = mic = sender = None
    state = {"active": False, "done": False}   # 끼어들기 시 서버 응답 취소 판단용

    session_kwargs = dict(
        modalities=[Modality.TEXT, Modality.AUDIO],
        instructions=instructions,
        voice=_build_voice(voice, lexicon_url),
        input_audio_format=InputAudioFormat.PCM16,
        output_audio_format=OutputAudioFormat.PCM16,
        turn_detection=AzureSemanticVadMultilingual(threshold=vad_threshold, prefix_padding_ms=300,
                                 silence_duration_ms=silence_ms, remove_filler_words=True),
        input_audio_echo_cancellation=AudioEchoCancellation(),
        input_audio_noise_reduction=AudioNoiseReduction(type="azure_deep_noise_suppression"),
    )
    if phrase_list:
        session_kwargs["input_audio_transcription"] = AudioInputTranscriptionOptions(
            model=transcription_model, language=language, phrase_list=phrase_list)
    if tools:
        session_kwargs["tools"] = tools
        session_kwargs["tool_choice"] = ToolChoiceLiteral.AUTO

    try:
        speaker = Speaker(fade_sec=fade_sec)
        loop = asyncio.get_running_loop()
        mic_q: asyncio.Queue[bytes] = asyncio.Queue()

        def mic_cb(indata, frames, time, status):
            loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

        mic = sd.RawInputStream(samplerate=24000, channels=1, dtype="int16",
                                blocksize=1200, latency="high", callback=mic_cb)

        async with connect(endpoint=AZURE_OPENAI_URI,
                           credential=AzureKeyCredential(AZURE_OPENAI_API_KEY),
                           model=model) as conn:
            await conn.session.update(session=RequestSession(**session_kwargs))
            speaker.start(); mic.start()
            print(intro)

            async def pump_mic():
                while True:
                    chunk = await mic_q.get()
                    await conn.input_audio_buffer.append(audio=base64.b64encode(chunk).decode())
            sender = asyncio.create_task(pump_mic())

            cur_gen = 0
            async for event in conn:
                if event.type == ServerEventType.RESPONSE_CREATED:
                    state["active"], state["done"] = True, False
                    cur_gen = speaker.gen                 # 이 응답의 세대 고정
                elif event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                    speaker.add(event.delta, cur_gen)     # 세대와 함께 add
                elif event.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DELTA:
                    print(event.delta, end="", flush=True)
                elif event.type == ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED:
                    print(f"\n🧑 {event.transcript}")
                elif event.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED:
                    print("\n🎧 음성 인식 시작...")
                    speaker.clear()                       # gen++ → 옛 delta 자동 차단
                    if state["active"] and not state["done"]:
                        try:
                            await conn.response.cancel()
                        except Exception as e:
                            if "no active response" not in str(e).lower():
                                print(f"\n(취소 경고: {e})")
                elif event.type == ServerEventType.RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE:
                    args = json.loads(event.arguments)
                    print(f"\n🔧 도구 호출: {event.name}({args})")
                    result = await on_tool(event.name, args) if on_tool else "도구가 없습니다."
                    await conn.conversation.item.create(
                        item=FunctionCallOutputItem(call_id=event.call_id, output=result))
                    await conn.response.create()
                elif event.type == ServerEventType.RESPONSE_DONE:
                    state["active"], state["done"] = False, True
                    print("\n✅ 응답 완료\n")
                elif event.type == ServerEventType.ERROR:
                    msg = getattr(event.error, "message", str(event.error))
                    if "no active response" not in msg.lower():
                        print("\n⚠️  ERROR:", msg)
    finally:
        if sender:
            sender.cancel()
        if mic:
            try: mic.stop(); mic.close()
            except Exception as e: print(f"마이크 종료 에러: {e}")
        if speaker:
            try: speaker.close()
            except Exception as e: print(f"스피커 종료 에러: {e}")

### (Demo) Voice-Live api -> S2S(realtime)

In [ ]:
await run_voicelive(
    model="gpt-realtime-1.5",
    #voice="marin",
    # voice="en-us-Ava3:DragonHDLatestNeural",
    voice="ko-kr-Sunhi:DragonHDLatestNeural",
    instructions="너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해",
)

### (Demo) Voice-Live api -> CaseCade(gpt-5.4)

In [ ]:
await run_voicelive(
    model=AZURE_OPENAI_CHAT_DEPLOYMENT,
    # voice="ko-kr-Sunhi:DragonHDLatestNeural", # 말하는 중간에 소리가 끊기는 현상(서버 AEC 작동) 아주 가끔 발생 하고 있음.
    # voice="ko-kr-Hyunsu:DragonHDLatestNeural",
    # voice="en-us-Andrew:DragonHDLatestNeural",
    voice="en-us-Ava3:DragonHDLatestNeural", #자연스럽지만 한글단어 발음 아님, 서버 AEC 아주 아주 가끔 안됨.
    instructions="너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해",
)

### 입력·출력 커스터마이징 — Phrase list & Custom lexicon

위 Voice Live 세션을 그대로 두면 일반 단어는 잘 처리하지만, **고유명사·브랜드명·전문용어** 같은 도메인 단어는 인식(입력)도 발음(출력)도 어긋나기 쉽다. Voice Live 는 세션 설정만으로 적용 가능한 가벼운 커스터마이징 두 가지를 제공한다.

| 구분 | 대상 | 효과 |
|------|------|------|
| **Phrase list** | 입력(STT) | 지정한 단어/구를 인식 후보로 가중 → 받아쓰기 정확도↑ |
| **Custom lexicon** | 출력(TTS) | 발음 사전(PLS/SSML)으로 특정 단어의 발음을 교정 |

### (Demo) Voice-Live api -> CaseCade(gpt-5.4+pharse+Lexicon)

In [ ]:
PHRASE_LIST = [
    "갤럭시",
    "갤럭시 제트 플립",
    "갤럭시 제트 폴드",
    "갤럭시 워치",
    "갤럭시 버즈",
    "갤럭시 북",
    "비스포크",
    "네오 큐엘이디",
    "큐엘이디",
    "더 프레임",
    "오디세이",
    "패밀리 허브",
]

# 영어 철자/약어는 다국어 보이스가 영어 발음으로 읽고, 숫자 약어는 숫자 정규화에 막혀
# lexicon 만으로는 불안정하다. → cascaded(LLM→TTS) 에선 instruction 으로 "한글 발음 표기"를
# 강제하는 게 가장 확실하다. (아래 매핑은 resources/lexicon.xml 의 단어들과 동일)
await run_voicelive(
    model=AZURE_OPENAI_CHAT_DEPLOYMENT,
    # voice="ko-kr-Sunhi:DragonHDLatestNeural",
    # voice="ko-kr-Hyunsu:DragonHDLatestNeural",
    # voice="en-us-Andrew:DragonHDLatestNeural",
    voice="en-us-Ava3:DragonHDLatestNeural", # 말하는 중에 소리가 안나오는 현상 가끔 발생
    lexicon_url=AZURE_SPEECH_LEXICON_URL,   # (참고용) 남겨두지만 핵심은 아래 instruction
    phrase_list=PHRASE_LIST,                # 입력 인식 가중
    instructions=(
        "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해"
        # "영어 철자·약어는 절대 그대로 쓰지 말고 항상 한글 발음으로 적어 말해. 다음 표기를 반드시 따른다: "
        # "Galaxy→갤럭시, Galaxy Z Flip→갤럭시 제트 플립, Galaxy Z Fold→갤럭시 제트 폴드, "
        # "Galaxy Buds→갤럭시 버즈, Galaxy Watch→갤럭시 워치, Galaxy Book→갤럭시 북, Ultra→울트라, "
        # "mm→미리, 360→삼육공, Tap S→탭 에쓰, Tap A→탭 에이, Bespoke→비스포크, "
        # "Neo QLED→네오 큐엘이디, QLED→큐엘이디, The Frame→더 프레임, Odyssey→오디세이, "
        # "Family Hub→패밀리 허브, 4K→포케이, 8K→팔케이, Q9000→큐구천. "
        # "검색 결과나 입력에 영어로 적혀 있어도 너는 위 한글 표기로 바꿔서 말해."
    ),
    vad_threshold=0.6,        # 잔향에 덜 민감하게
    silence_ms=600,
    intro="🎙️  말해보세요. '갤럭시 Z 플립', '비스포크', '4K' 같은 제품명/약어로 발음을 확인해보세요.\n",
)

## gpt-realtime
OpenAI 가 개발한 실시간 음성-음성 대화형 AI 모델이다. STT 나 TTS 의 중간 과정없이 voice 인터페이스를 제공하고 사람처럼 자연스러운 대화가 가능하다.

아래 코드는 OpenAI realtime WebSocket 을 직접 다루는 간단한 voice 챗봇이다. 이 경로는 **서버측 에코 제거(AEC)가 없다.** 헤드폰 없이 스피커로 실행하면 AI 음성이 마이크로 되먹어(에코) 자기 목소리에 또 응답하거나 재생이 끊기므로, 코드에서는 **AI 가 말하는 동안 마이크 전송을 잠시 끊는 하프듀플렉스(half-duplex) 마이크 게이팅**으로 에코 피드백을 차단한다. 대신 재생 중 끼어들기(barge-in)는 비활성된다. 또한 네트워크 지터로 인한 재생 끊김을 줄이기 위해 **지터버퍼 스피커**(prebuffer 후 재생)를 사용한다.

완전한 끼어들기가 필요하면 헤드폰을 쓰거나, 서버 AEC 가 내장된 아래 **Voice Live** 를 사용한다.

### 동작 원리 (블록도)

```text
   🎙️ 마이크 (PCM 24kHz)
        │   ⚠️ AI 재생 중에는 마이크 전송을 중단(하프듀플렉스 게이팅) → 에코 차단
        │  ▶ input_audio_buffer.append      (client ──WS──▶ server)
        ▼
  ┌─ gpt-realtime (서버) ───────────────────────────────────────
  │   ① Server VAD     : 발화 시작/종료를 감지해 대화 턴을 분리
  │   ② Speech-to-Speech: 음성을 직접 이해·추론하고 음성으로 직접
  │                       생성 (STT·TTS 중간 변환 단계가 없음)
  │   ③ Transcript     : 입력/출력 텍스트도 함께 스트리밍
  └──────────────────────────────────────────────────────────────
        │
        │  ◀ response.output_audio.delta    (server ──WS──▶ client)
        ▼
   🔊 스피커 (PCM 24kHz, 지터버퍼로 재생)

   ↕ 프로토콜 자체는 하나의 WebSocket 양방향(full-duplex) 스트림 → 낮은 지연(low latency)
     단, 서버 AEC 가 없어 스피커 데모에서는 클라이언트가 마이크를 게이팅하므로
     재생 중 끼어들기(barge-in)는 동작하지 않는다(헤드폰 사용 시 자연스러운 대화 가능).
```

**아래 코드의 이벤트가 위 흐름에 그대로 대응한다.**

| 단계 | Realtime 이벤트 / 처리 |
|------|-----------------|
| 마이크 → 서버 전송 | `input_audio_buffer.append` (AI 재생 중엔 전송 차단) |
| 사용자 발화 감지 | `input_audio_buffer.speech_started` |
| 응답 오디오 수신(재생) | `response.output_audio.delta` (지터버퍼 → 스피커) |
| 음성 인식 결과 | `conversation.item.input_audio_transcription.completed` |
| 응답 종료 | `response.done` |


### (Demo) gpt-realtime-2.0 끼어들기X

In [ ]:
import asyncio
import base64
import threading
import time

import sounddevice as sd
from openai import AsyncOpenAI

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 지터 버퍼 스피커 (재생 끊김 완화)
# - 네트워크로 오는 오디오 delta 는 버스트로 도착하므로, 일정량(prebuffer)을
#   모은 뒤 재생을 시작하고 고정 블록(20ms)으로 흘려보내 언더런(끊김)을 줄인다.
class Speaker:
    def __init__(self, samplerate=24000, prebuffer_ms=300):
        self.sr = samplerate
        self.buf = bytearray()
        self.pos = 0                     # 읽기 포인터(소비한 바이트)
        self.lock = threading.Lock()
        self.playing = False
        self.prebuffer = int(samplerate * 2 * prebuffer_ms / 1000)  # 24kHz·16bit
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=480, latency="high", callback=self._cb,   # 20ms 고정 블록
        )

    def _avail(self):
        return len(self.buf) - self.pos

    def pending_bytes(self):
        # 아직 재생되지 않고 버퍼에 남아 있는 오디오 바이트 수 (마이크 게이팅 판단용)
        with self.lock:
            return self._avail()

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        # PortAudio 콜백은 별도의 스레드에서 호출되므로, 버퍼 접근을 락으로 보호
        with self.lock:
            # 처음 시작 또는 완전히 바닥난 뒤에만 선버퍼링(prebuffer) 대기
            if not self.playing:
                if self._avail() < self.prebuffer:
                    outdata[:] = b"\x00" * need
                    return
                self.playing = True

            n = min(need, self._avail())
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n])
            self.pos += n

            if n < need:
                outdata[n:] = b"\x00" * (need - n)
                # 짧은 공백이면 playing 유지(무음만 채움), 완전히 비면 그때만 재대기
                if self._avail() == 0:
                    self.playing = False

            # 다 읽은 앞부분 주기적으로 회수(O(n) 삭제를 자주 안 하도록)
            if self.pos > 2_000_000:
                del self.buf[:self.pos]
                self.pos = 0

    def add(self, pcm: bytes):
        # PCM 데이터를 버퍼에 추가하는 메서드. 콜백과 동시에 호출될 수 있으므로 락으로 보호
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        # 버퍼 초기화(끼어들기 시 재생 중단). 콜백과 동시에 호출될 수 있으므로 락으로 보호
        with self.lock:
            del self.buf[:]
            self.pos = 0
            self.playing = False

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()


# ── 에코 대책: 하프듀플렉스 마이크 게이팅 ──────────────────────────────
# 이 셀은 직접 WebSocket 이라 서버측 에코 제거(AEC)가 없다. 스피커로 데모하면
# AI 음성이 마이크로 되먹어(에코) → 서버 VAD 가 사용자 발화로 오인 → 자기 목소리에
# 또 응답(반복)하거나, 가짜 speech_started 로 재생을 끊어버린다(끊김).
# 헤드폰 없이 스피커만으로 쓰려면, "AI 가 말하는 동안 마이크 전송을 끊어"
# 서버가 자기 목소리를 아예 못 듣게 한다(= 끼어들기는 포기, 안정성 확보).
MIC_RESUME_GUARD_S = 0.4  # 스피커 버퍼가 빈 뒤에도 공기 전달 잔향이 남으므로 추가 대기

# ── 토큰 사용량 누적 집계 ─────────────────────────────────────────────
# gpt-realtime 은 response.done 의 response.usage 로 그 턴의 토큰량을 준다(per-turn).
# 세션 전체를 보려면 턴마다 누적한다.
usage_totals = {
    "input": 0, "input_text": 0, "input_audio": 0, "input_cached": 0,
    "output": 0, "output_text": 0, "output_audio": 0, "total": 0,
}

def accumulate_usage(usage):
    if usage is None:
        return
    u = usage.model_dump() if hasattr(usage, "model_dump") else dict(usage)
    it = u.get("input_token_details") or {}
    ot = u.get("output_token_details") or {}
    usage_totals["input"]        += u.get("input_tokens", 0) or 0
    usage_totals["output"]       += u.get("output_tokens", 0) or 0
    usage_totals["total"]        += u.get("total_tokens", 0) or 0
    usage_totals["input_text"]   += it.get("text_tokens", 0) or 0
    usage_totals["input_audio"]  += it.get("audio_tokens", 0) or 0
    usage_totals["input_cached"] += it.get("cached_tokens", 0) or 0
    usage_totals["output_text"]  += ot.get("text_tokens", 0) or 0
    usage_totals["output_audio"] += ot.get("audio_tokens", 0) or 0

def print_usage_totals():
    print(
        f"📊 누적 토큰: 총 {usage_totals['total']:,} | "
        f"입력 {usage_totals['input']:,} "
        f"(텍스트 {usage_totals['input_text']:,} / 오디오 {usage_totals['input_audio']:,} / 캐시 {usage_totals['input_cached']:,}) | "
        f"출력 {usage_totals['output']:,} "
        f"(텍스트 {usage_totals['output_text']:,} / 오디오 {usage_totals['output_audio']:,})"
    )

def usage_to_dict(usage):
    # gpt-realtime 의 이번 턴 usage 를 표시용 dict 로 변환(누적하지 않음)
    d = dict.fromkeys(usage_totals, 0)
    if usage is None:
        return d
    u = usage.model_dump() if hasattr(usage, "model_dump") else dict(usage)
    it = u.get("input_token_details") or {}
    ot = u.get("output_token_details") or {}
    d["input"]        = u.get("input_tokens", 0) or 0
    d["output"]       = u.get("output_tokens", 0) or 0
    d["total"]        = u.get("total_tokens", 0) or 0
    d["input_text"]   = it.get("text_tokens", 0) or 0
    d["input_audio"]  = it.get("audio_tokens", 0) or 0
    d["input_cached"] = it.get("cached_tokens", 0) or 0
    d["output_text"]  = ot.get("text_tokens", 0) or 0
    d["output_audio"] = ot.get("audio_tokens", 0) or 0
    return d

def print_turn_usage(turn):
    print(
        f"🔹 이번 턴: 총 {turn['total']:,} | "
        f"입력 {turn['input']:,} "
        f"(텍스트 {turn['input_text']:,} / 오디오 {turn['input_audio']:,} / 캐시 {turn['input_cached']:,}) | "
        f"출력 {turn['output']:,} "
        f"(텍스트 {turn['output_text']:,} / 오디오 {turn['output_audio']:,})"
    )

# ── 대화 녹음(합본 WAV) + 전사 텍스트 저장 ─────────────────────────────
# 사용자·AI 오디오를 하나의 WAV 로 합본한다. 반이중/턴테이킹 구조라 도착 순서대로
# 이어붙이면 대화 타임라인이 되고, 샘플레이트가 다르면 out_rate 로 선형 리샘플한다.
import wave
import numpy as np

class ConversationRecorder:
    def __init__(self, out_rate=24000):
        self.out_rate = out_rate
        self.audio = bytearray()
        self.lines = []            # 전사 텍스트(시간순)

    def add_audio(self, pcm: bytes, src_rate: int):
        if not pcm:
            return
        if src_rate != self.out_rate:
            a = np.frombuffer(pcm, dtype=np.int16).astype(np.float32)
            n = int(len(a) * self.out_rate / src_rate)
            if n <= 0:
                return
            a = np.interp(
                np.linspace(0, 1, n, endpoint=False),
                np.linspace(0, 1, len(a), endpoint=False), a,
            ).astype(np.int16)
            pcm = a.tobytes()
        self.audio.extend(pcm)

    def add_text(self, who: str, text: str):
        if text and text.strip():
            self.lines.append(f"{who}: {text.strip()}")

    def save(self, base="conversation"):
        if self.audio:
            with wave.open(f"{base}.wav", "wb") as w:
                w.setnchannels(1); w.setsampwidth(2); w.setframerate(self.out_rate)
                w.writeframes(bytes(self.audio))
            secs = len(self.audio) / 2 / self.out_rate
            print(f"💾 대화 합본 저장: {base}.wav ({secs:.1f}s)")
        if self.lines:
            with open(f"{base}.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(self.lines) + "\n")
            print(f"💾 전사 저장: {base}.txt ({len(self.lines)}줄)")

recorder = ConversationRecorder()

# PortAudio 초기화 및 재설정
reset_portaudio()

# OpenAI 클라이언트 초기화
client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.replace("https://", "wss://").rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)
speaker, mic, sender = None, None, None
try:
    # 스피커와 마이크 초기화
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    # 마이크 콜백 함수: PCM 데이터를 받아서 asyncio 큐에 넣음
    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    # 마이크 스트림 설정: 24000Hz, 모노, 16비트 PCM, 콜백 함수 등록
    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16", blocksize=0, callback=mic_cb,
    )

    # OpenAI 실시간 연결 설정 및 이벤트 루프
    async with client.realtime.connect(model=AZURE_OPENAI_REALTIME_DEPLOYMENT) as conn:
        await conn.session.update(session={
            "type": "realtime",
            "instructions": "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해",
            "output_modalities": ["audio"],
            "audio": {
                "input": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "transcription": {"model": "gpt-4o-transcribe", "language": "ko"}, # 실시간 음성 인식
                    "turn_detection": {
                        "type": "server_vad",   # 서버에서 음성 활동 감지(VAD) 사용
                        "threshold": 0.5,   # VAD 민감도 (0.0~1.0, 낮을수록 민감)
                        "prefix_padding_ms": 300,   # 음성 시작 전 패딩
                        "silence_duration_ms": 500, # 음성 종료로 간주할 무음 지속 시간
                        "create_response": True,
                        # 마이크 게이팅으로 재생 중엔 입력이 없으므로 끼어들기는 사실상 비활성.
                        "interrupt_response": True,
                    },
                },
                "output": {
                    "voice": "marin",
                    "format": {"type": "audio/pcm", "rate": 24000},
                },
            },
        })
        speaker.start()
        mic.start()
        print("🎙️  말해보세요.\n")
        print("ℹ️  스피커 데모용 에코 차단 모드: AI 가 말하는 동안엔 마이크가 잠시 멈춥니다.\n")

        # 마이크에서 PCM 데이터를 읽어서 OpenAI로 전송하는 비동기 태스크
        # - AI 가 재생 중(또는 직후 가드 구간)이면 전송을 건너뛰어 에코 피드백을 차단
        mic_gate = {"resume_at": 0.0}

        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                now = time.monotonic()
                # 스피커가 아직 재생할 오디오를 갖고 있으면 → 전송 차단 + 가드 갱신
                if speaker.pending_bytes() > 0:
                    mic_gate["resume_at"] = now + MIC_RESUME_GUARD_S
                    continue
                # 재생이 끝나도 잔향이 남는 가드 구간 동안은 계속 차단
                if now < mic_gate["resume_at"]:
                    continue
                recorder.add_audio(chunk, 24000)   # 전송(=게이팅 통과)한 사용자 음성만 녹음
                await conn.input_audio_buffer.append(
                    audio=base64.b64encode(chunk).decode(),
                )

        sender = asyncio.create_task(pump_mic())
        ai_text = []   # 이번 턴 AI 전사 누적(녹음 텍스트용)
        # OpenAI 이벤트 처리 루프
        async for event in conn:
            # 이벤트 타입에 따라 스피커에 PCM 데이터를 추가하거나, 인식된 텍스트를 출력하거나, 에러를 처리
            if event.type == "response.output_audio.delta":
                pcm = base64.b64decode(event.delta)
                speaker.add(pcm)
                recorder.add_audio(pcm, 24000)   # AI 음성 녹음
            # AI 응답 전사(텍스트) 스트리밍
            elif event.type == "response.output_audio_transcript.delta":
                print(event.delta, end="", flush=True)
                ai_text.append(event.delta)
            # 사용자 음성 인식 결과
            elif event.type == "conversation.item.input_audio_transcription.completed":
                print(f"\n🧑 {event.transcript}")
                recorder.add_text("🧑 사용자", event.transcript)
            elif event.type == "conversation.item.input_audio_transcription.failed":
                print(f"\n\U0001f9d1 음성 인식 실패: {event.error}")
            # 사용자가 말하기 시작 → 재생 중단(끼어들기)
            elif event.type == "input_audio_buffer.speech_started":
                print("\n🎧 음성 인식 시작...")
                speaker.clear()
            # 응답 완료
            elif event.type == "response.done":
                usage_obj = getattr(event.response, "usage", None)
                accumulate_usage(usage_obj)
                recorder.add_text("🤖 AI", "".join(ai_text)); ai_text = []
                print("\n✅ 응답 완료")
                print_turn_usage(usage_to_dict(usage_obj))   # 이번 턴
                print_usage_totals()                          # 세션 누적
                print()
            elif event.type == "error":
                print("\n⚠️  ERROR:", event.error)
finally:
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")
    print("\n=== 세션 누적 토큰 사용량 ===")
    print_usage_totals()
    recorder.save("gpt_realtime_speaker")

### (Demo) gpt-realtime-2.0 끼어들기 O (w/ headset)

헤드셋을 쓰면 에코가 없으므로 마이크 게이팅 없이 **풀듀플렉스 + 진짜 끼어들기(barge-in)** 가 된다. gpt-realtime-2.0 을 직접 WebSocket 으로 연결하고, `interrupt_response: True` + `speech_started` 시 `speaker.clear()` 로 AI 발화 도중 끊고 들어갈 수 있다. (서버 AEC 가 없는 직접 경로지만, 헤드셋이 에코 경로를 물리적으로 차단)

| | 스피커 버전 | 이 헤드셋 버전 |
|---|---|---|
| 마이크 게이팅 | 있음(에코 차단용) | **없음 — 항상 켜짐** |
| `interrupt_response` | 사실상 무력 | **True (진짜 끼어들기)** |
| `speech_started` | 에코 무시 | **즉시 `speaker.clear()`** |
| 전제 | 에코 환경(스피커) | **헤드셋(에코 없음)** |

In [ ]:
import asyncio
import base64
import threading

import sounddevice as sd
from openai import AsyncOpenAI

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 스피커 — 끼어들기 시 clear() 로 재생 즉시 중단
class Speaker:
    def __init__(self, samplerate=24000, blocksize=1200):
        self.buf, self.pos = bytearray(), 0
        self.lock = threading.Lock()
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=blocksize, callback=self._cb)

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            avail = len(self.buf) - self.pos
            n = min(need, avail)
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n]); self.pos += n
            if n < need:
                outdata[n:] = b"\x00" * (need - n)
            if self.pos > 2_000_000:
                del self.buf[:self.pos]; self.pos = 0

    def add(self, pcm):
        with self.lock: self.buf.extend(pcm)
    def clear(self):
        with self.lock: del self.buf[:]; self.pos = 0
    def start(self): self.stream.start()
    def close(self):
        try: self.stream.stop()
        finally: self.stream.close()


# ── 토큰 사용량 누적 집계 (끼어들기X 셀과 동일) ─────────────────────────
# gpt-realtime 은 매 응답이 끝날 때 response.done 이벤트의 response.usage 로
# 그 응답 한 턴의 토큰 사용량을 준다. 세션 전체 비용을 보려면 턴마다 누적해야 한다.
usage_totals = {
    "input": 0, "input_text": 0, "input_audio": 0, "input_cached": 0,
    "output": 0, "output_text": 0, "output_audio": 0, "total": 0,
}

def accumulate_usage(usage):
    # usage 는 pydantic 객체(SDK) — model_dump 로 dict 화 후 안전하게 누적
    if usage is None:
        return
    u = usage.model_dump() if hasattr(usage, "model_dump") else dict(usage)
    it = u.get("input_token_details") or {}
    ot = u.get("output_token_details") or {}
    usage_totals["input"]        += u.get("input_tokens", 0) or 0
    usage_totals["output"]       += u.get("output_tokens", 0) or 0
    usage_totals["total"]        += u.get("total_tokens", 0) or 0
    usage_totals["input_text"]   += it.get("text_tokens", 0) or 0
    usage_totals["input_audio"]  += it.get("audio_tokens", 0) or 0
    usage_totals["input_cached"] += it.get("cached_tokens", 0) or 0
    usage_totals["output_text"]  += ot.get("text_tokens", 0) or 0
    usage_totals["output_audio"] += ot.get("audio_tokens", 0) or 0

def print_usage_totals():
    print(
        f"📊 누적 토큰: 총 {usage_totals['total']:,} | "
        f"입력 {usage_totals['input']:,} "
        f"(텍스트 {usage_totals['input_text']:,} / 오디오 {usage_totals['input_audio']:,} / 캐시 {usage_totals['input_cached']:,}) | "
        f"출력 {usage_totals['output']:,} "
        f"(텍스트 {usage_totals['output_text']:,} / 오디오 {usage_totals['output_audio']:,})"
    )

def usage_to_dict(usage):
    # gpt-realtime 의 이번 턴 usage 를 표시용 dict 로 변환(누적하지 않음)
    d = dict.fromkeys(usage_totals, 0)
    if usage is None:
        return d
    u = usage.model_dump() if hasattr(usage, "model_dump") else dict(usage)
    it = u.get("input_token_details") or {}
    ot = u.get("output_token_details") or {}
    d["input"]        = u.get("input_tokens", 0) or 0
    d["output"]       = u.get("output_tokens", 0) or 0
    d["total"]        = u.get("total_tokens", 0) or 0
    d["input_text"]   = it.get("text_tokens", 0) or 0
    d["input_audio"]  = it.get("audio_tokens", 0) or 0
    d["input_cached"] = it.get("cached_tokens", 0) or 0
    d["output_text"]  = ot.get("text_tokens", 0) or 0
    d["output_audio"] = ot.get("audio_tokens", 0) or 0
    return d

def print_turn_usage(turn):
    print(
        f"🔹 이번 턴: 총 {turn['total']:,} | "
        f"입력 {turn['input']:,} "
        f"(텍스트 {turn['input_text']:,} / 오디오 {turn['input_audio']:,} / 캐시 {turn['input_cached']:,}) | "
        f"출력 {turn['output']:,} "
        f"(텍스트 {turn['output_text']:,} / 오디오 {turn['output_audio']:,})"
    )

# ── 대화 녹음(합본 WAV) + 전사 텍스트 저장 ─────────────────────────────
# 사용자·AI 오디오를 하나의 WAV 로 합본한다. 반이중/턴테이킹 구조라 도착 순서대로
# 이어붙이면 대화 타임라인이 되고, 샘플레이트가 다르면 out_rate 로 선형 리샘플한다.
import wave
import numpy as np

class ConversationRecorder:
    def __init__(self, out_rate=24000):
        self.out_rate = out_rate
        self.audio = bytearray()
        self.lines = []            # 전사 텍스트(시간순)

    def add_audio(self, pcm: bytes, src_rate: int):
        if not pcm:
            return
        if src_rate != self.out_rate:
            a = np.frombuffer(pcm, dtype=np.int16).astype(np.float32)
            n = int(len(a) * self.out_rate / src_rate)
            if n <= 0:
                return
            a = np.interp(
                np.linspace(0, 1, n, endpoint=False),
                np.linspace(0, 1, len(a), endpoint=False), a,
            ).astype(np.int16)
            pcm = a.tobytes()
        self.audio.extend(pcm)

    def add_text(self, who: str, text: str):
        if text and text.strip():
            self.lines.append(f"{who}: {text.strip()}")

    def save(self, base="conversation"):
        if self.audio:
            with wave.open(f"{base}.wav", "wb") as w:
                w.setnchannels(1); w.setsampwidth(2); w.setframerate(self.out_rate)
                w.writeframes(bytes(self.audio))
            secs = len(self.audio) / 2 / self.out_rate
            print(f"💾 대화 합본 저장: {base}.wav ({secs:.1f}s)")
        if self.lines:
            with open(f"{base}.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(self.lines) + "\n")
            print(f"💾 전사 저장: {base}.txt ({len(self.lines)}줄)")

recorder = ConversationRecorder()

reset_portaudio()

# gpt-realtime 은 WebSocket(wss) 직접 연결
client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.replace("https://", "wss://").rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)
speaker, mic, sender = None, None, None
try:
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16",
        blocksize=2400, latency="high", callback=mic_cb,
    )

    async with client.realtime.connect(model=AZURE_OPENAI_REALTIME_DEPLOYMENT) as conn:
        await conn.session.update(session={
            "type": "realtime",
            "instructions": "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해",
            "output_modalities": ["audio"],
            "audio": {
                "input": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "transcription": {"model": "gpt-4o-transcribe", "language": "ko"},
                    "turn_detection": {
                        "type": "server_vad",
                        "threshold": 0.5,
                        "prefix_padding_ms": 300,
                        "silence_duration_ms": 500,
                        "create_response": True,
                        "interrupt_response": True,   # ← 끼어들기 ON (헤드셋이라 에코 오작동 없음)
                    },
                },
                "output": {
                    "voice": "marin",
                    "format": {"type": "audio/pcm", "rate": 24000},
                },
            },
        })
        speaker.start()
        mic.start()
        print("\U0001f3a7 헤드셋 착용 후 말해보세요. AI 발화 도중에 끼어들 수 있어요.\n")

        # 헤드셋 → 에코 없음 → 마이크를 항상 켜둔다(게이팅 X = 진짜 full-duplex)
        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                recorder.add_audio(chunk, 24000)
                await conn.input_audio_buffer.append(audio=base64.b64encode(chunk).decode())

        sender = asyncio.create_task(pump_mic())
        ai_text = []
        async for event in conn:
            if event.type == "response.output_audio.delta":
                pcm = base64.b64decode(event.delta)
                speaker.add(pcm)
                recorder.add_audio(pcm, 24000)
            elif event.type == "response.output_audio_transcript.delta":
                print(event.delta, end="", flush=True)
                ai_text.append(event.delta)
            elif event.type == "conversation.item.input_audio_transcription.completed":
                print(f"\n\U0001f9d1 {event.transcript}")
                recorder.add_text("🧑 사용자", event.transcript)
            elif event.type == "conversation.item.input_audio_transcription.failed":
                print(f"\n\U0001f9d1 음성 인식 실패: {event.error}")
            # 끼어들기: 재생 즉시 중단(서버는 interrupt_response=True 로 진행 중 응답을 취소)
            elif event.type == "input_audio_buffer.speech_started":
                print("\n\U0001f3a7 (끼어들기) 음성 인식 시작...")
                speaker.clear()
            # 응답 완료 → 이 턴의 usage 를 누적하고 세션 누적치를 출력
            elif event.type == "response.done":
                usage_obj = getattr(event.response, "usage", None)
                accumulate_usage(usage_obj)
                recorder.add_text("🤖 AI", "".join(ai_text)); ai_text = []
                print("\n✅ 응답 완료")
                print_turn_usage(usage_to_dict(usage_obj))   # 이번 턴
                print_usage_totals()                          # 세션 누적
                print()
            elif event.type == "error":
                print("\n⚠️  ERROR:", event.error)
finally:
    if sender:
        sender.cancel()
    if mic:
        try: mic.stop(); mic.close()
        except Exception as e: print(f"마이크 종료 에러: {e}")
    if speaker:
        try: speaker.close()
        except Exception as e: print(f"스피커 종료 에러: {e}")
    # 세션 종료 시 최종 누적 토큰 사용량 출력
    print("\n=== 세션 누적 토큰 사용량 ===")
    print_usage_totals()
    recorder.save("gpt_realtime_headset")


## gemini-3.1-flash-live
Google 가 개발한 실시간 음성-음성(멀티모달 Live) 대화형 AI 모델이다. `gpt-realtime` 과 마찬가지로 STT·TTS 중간 단계 없이 음성을 직접 이해·생성하며, `google-genai` SDK 의 **Live API** 로 하나의 WebSocket 세션을 통해 스트리밍한다.

앞의 `gpt-realtime` 코드와 큰 흐름(마이크 → 서버 → 스피커)은 같지만, SDK/이벤트/샘플레이트에서 몇 가지 구현 차이가 있다.

| 구분 | gpt-realtime | gemini-3.1-flash-live |
|------|--------------|------------------------|
| SDK / 연결 | `openai` · OpenAI Realtime WebSocket | `google-genai` · Gemini Live API (`client.aio.live.connect`) |
| 입력 오디오 | PCM 24kHz | **PCM 16kHz** (Gemini 입력 요구사항) |
| 출력 오디오 | PCM 24kHz | PCM 24kHz |
| 오디오 전송 | `input_audio_buffer.append(base64)` | `send_realtime_input(audio=Blob(...))` (raw bytes) |
| 턴 감지(VAD) | `server_vad` 명시 설정 | 서버 자동 VAD (기본 활성) |
| 끼어들기 신호 | `input_audio_buffer.speech_started` | `server_content.interrupted` |
| 전사(transcription) | `conversation.item.input_audio_transcription.*` | `server_content.input/output_transcription` (스트리밍) |
| 보이스 | `marin` 등 OpenAI 보이스 | `Charon`, `Puck`, `Kore`, `Aoede` 등 Gemini 보이스 |
| 토큰 사용량 | `response.done` 의 per-turn usage | `usage_metadata` (세션 누적 스냅샷) |

- <small> gpt-realtime 와 마찬가지로 Gemini Live 도 클라이언트측 서버 AEC 가 없어, 스피커로 데모하면 AI 음성이 마이크로 되먹는다. 아래에서도 **스피커용 하프듀플렉스 마이크 게이팅** 버전과 **헤드셋용 끼어들기(barge-in)** 버전을 나눠 제공한다.</small>


### 동작 원리 (블록도)

```text
   🎙️ 마이크 (PCM 16kHz)   ← Gemini Live 입력은 16kHz
        │   ⚠️ AI 재생 중에는 마이크 전송을 중단(하프듀플렉스 게이팅) → 에코 차단
        │  ▶ send_realtime_input(audio=Blob)   (client ──WS──▶ server)
        ▼
  ┌─ gemini-3.1-flash-live (서버) ────────────────────────────────
  │   ① 자동 VAD        : 발화 시작/종료를 감지해 대화 턴을 분리
  │   ② Speech-to-Speech: 음성을 직접 이해·추론하고 음성으로 직접 생성
  │   ③ Transcription   : 입력/출력 텍스트도 함께 스트리밍
  └──────────────────────────────────────────────────────────────
        │
        │  ◀ msg.data (inline audio)   (server ──WS──▶ client)
        ▼
   🔊 스피커 (PCM 24kHz, 지터버퍼로 재생)   ← Gemini Live 출력은 24kHz

   ↕ 하나의 WebSocket 양방향(full-duplex) 스트림 → 낮은 지연(low latency)
     단, 서버 AEC 가 없어 스피커 데모에서는 클라이언트가 마이크를 게이팅하므로
     재생 중 끼어들기(barge-in)는 동작하지 않는다(헤드셋 사용 시 자연스러운 대화 가능).
```

**아래 코드의 처리가 위 흐름에 그대로 대응한다.**

| 단계 | Gemini Live 처리 |
|------|-----------------|
| 마이크 → 서버 전송 | `session.send_realtime_input(audio=Blob(...))` (AI 재생 중엔 전송 차단) |
| 응답 오디오 수신(재생) | `msg.data` (지터버퍼 → 스피커) |
| 사용자/AI 음성 인식 결과 | `msg.server_content.input_transcription` / `output_transcription` |
| 끼어들기 감지 | `msg.server_content.interrupted` |
| 응답 종료 | `msg.server_content.turn_complete` |


In [ ]:
# gemini-3.1-flash-live 실습용 패키지 설치
%pip -q install -U google-genai sounddevice==0.5.5

# --- .env 에 아래 값을 추가하세요 ---
#   GEMINI_API_KEY=<your key>
#   GEMINI_LIVE_MODEL=gemini-3.1-flash-live   # (선택) 미지정 시 기본값 사용
import os
from dotenv import load_dotenv

load_dotenv(override=True)

GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY")
GEMINI_LIVE_MODEL = os.getenv("GEMINI_LIVE_MODEL", "gemini-3.1-flash-live-preview")
assert GEMINI_API_KEY, "GEMINI_API_KEY 가 .env 에 설정되어 있지 않습니다."

# 두 데모가 공유하는 시스템 프롬프트
INSTRUCTIONS = (
    "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. "
    "사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, "
    "실제 사람이 말하듯 자연스럽고 간결하게 표현해"
)
print(f"✅ Gemini Live 모델: {GEMINI_LIVE_MODEL}")


### (Demo) gemini-3.1-flash-live 끼어들기X

In [ ]:
import asyncio
import threading
import time

import sounddevice as sd
from google import genai
from google.genai import types

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 지터 버퍼 스피커 (재생 끊김 완화) — Gemini Live 출력은 24kHz PCM
# - 네트워크로 오는 오디오는 버스트로 도착하므로, 일정량(prebuffer)을 모은 뒤
#   재생을 시작하고 고정 블록(20ms)으로 흘려보내 언더런(끊김)을 줄인다.
class Speaker:
    def __init__(self, samplerate=24000, prebuffer_ms=300):
        self.sr = samplerate
        self.buf = bytearray()
        self.pos = 0                     # 읽기 포인터(소비한 바이트)
        self.lock = threading.Lock()
        self.playing = False
        self.prebuffer = int(samplerate * 2 * prebuffer_ms / 1000)  # 24kHz·16bit
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=480, latency="high", callback=self._cb,   # 20ms 고정 블록
        )

    def _avail(self):
        return len(self.buf) - self.pos

    def pending_bytes(self):
        # 아직 재생되지 않고 버퍼에 남아 있는 오디오 바이트 수 (마이크 게이팅 판단용)
        with self.lock:
            return self._avail()

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        # PortAudio 콜백은 별도의 스레드에서 호출되므로, 버퍼 접근을 락으로 보호
        with self.lock:
            # 처음 시작 또는 완전히 바닥난 뒤에만 선버퍼링(prebuffer) 대기
            if not self.playing:
                if self._avail() < self.prebuffer:
                    outdata[:] = b"\x00" * need
                    return
                self.playing = True

            n = min(need, self._avail())
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n])
            self.pos += n

            if n < need:
                outdata[n:] = b"\x00" * (need - n)
                # 짧은 공백이면 playing 유지(무음만 채움), 완전히 비면 그때만 재대기
                if self._avail() == 0:
                    self.playing = False

            # 다 읽은 앞부분 주기적으로 회수(O(n) 삭제를 자주 안 하도록)
            if self.pos > 2_000_000:
                del self.buf[:self.pos]
                self.pos = 0

    def add(self, pcm: bytes):
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        # 버퍼 초기화(끼어들기 시 재생 중단)
        with self.lock:
            del self.buf[:]
            self.pos = 0
            self.playing = False

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()


# ── 에코 대책: 하프듀플렉스 마이크 게이팅 ──────────────────────────────
# gpt-realtime 과 동일하게 Gemini Live 도 클라이언트측 서버 AEC 가 없다.
# 스피커로 데모하면 AI 음성이 마이크로 되먹어 서버 자동 VAD 가 사용자 발화로
# 오인 → 자기 목소리에 재응답/재생 끊김이 생긴다. 그래서 "AI 가 말하는 동안엔
# 마이크 전송을 끊어" 서버가 자기 목소리를 아예 못 듣게 한다(= 끼어들기 포기, 안정성 확보).
MIC_RESUME_GUARD_S = 0.4  # 스피커 버퍼가 빈 뒤에도 공기 전달 잔향이 남으므로 추가 대기

# ── 토큰 사용량 누적 집계 ───────────────────────────────────────────────
# Gemini 의 usage_metadata 는 "이번 턴(1회 추론)"의 사용량(per-request)이다.
# (대화 이력이 매 턴 입력 프롬프트에 다시 포함되므로 input 값이 턴마다 커 보이지만,
#  그건 누적이 아니라 그 턴의 입력 크기다.) 세션 전체 비용을 보려면 gpt-realtime
# 처럼 턴마다 값을 "더해야" 한다.
usage_totals = {
    "input": 0, "input_text": 0, "input_audio": 0, "input_cached": 0,
    "output": 0, "output_text": 0, "output_audio": 0, "total": 0,
}

def _modality(d):
    return str(getattr(d.modality, "name", d.modality)).upper()

def _extract_usage(u):
    # 이번 턴의 usage_metadata 를 dict 로 정규화 (per-turn 스냅샷)
    snap = dict.fromkeys(usage_totals, 0)
    if u is None:
        return snap
    snap["input"]        = u.prompt_token_count or 0
    snap["output"]       = u.response_token_count or 0
    snap["total"]        = u.total_token_count or 0
    snap["input_cached"] = getattr(u, "cached_content_token_count", 0) or 0
    for d in (u.prompt_tokens_details or []):
        if "AUDIO" in _modality(d): snap["input_audio"] = d.token_count or 0
        elif "TEXT" in _modality(d): snap["input_text"] = d.token_count or 0
    for d in (u.response_tokens_details or []):
        if "AUDIO" in _modality(d): snap["output_audio"] = d.token_count or 0
        elif "TEXT" in _modality(d): snap["output_text"] = d.token_count or 0
    return snap

def accumulate_turn(u):
    # 한 턴이 끝날 때 그 턴의 usage 를 세션 누적치에 더하고 그 턴 값을 돌려준다
    snap = _extract_usage(u)
    for k in usage_totals:
        usage_totals[k] += snap[k]
    return snap

def print_turn_usage(turn):
    print(
        f"🔹 이번 턴: 총 {turn['total']:,} | "
        f"입력 {turn['input']:,} "
        f"(텍스트 {turn['input_text']:,} / 오디오 {turn['input_audio']:,} / 캐시 {turn['input_cached']:,}) | "
        f"출력 {turn['output']:,} "
        f"(텍스트 {turn['output_text']:,} / 오디오 {turn['output_audio']:,})"
    )

def print_usage_totals():
    print(
        f"📊 누적 토큰: 총 {usage_totals['total']:,} | "
        f"입력 {usage_totals['input']:,} "
        f"(텍스트 {usage_totals['input_text']:,} / 오디오 {usage_totals['input_audio']:,} / 캐시 {usage_totals['input_cached']:,}) | "
        f"출력 {usage_totals['output']:,} "
        f"(텍스트 {usage_totals['output_text']:,} / 오디오 {usage_totals['output_audio']:,})"
    )
# ── 대화 녹음(합본 WAV) + 전사 텍스트 저장 ─────────────────────────────
# 사용자·AI 오디오를 하나의 WAV 로 합본한다. 반이중/턴테이킹 구조라 도착 순서대로
# 이어붙이면 대화 타임라인이 되고, 샘플레이트가 다르면 out_rate 로 선형 리샘플한다.
import wave
import numpy as np

class ConversationRecorder:
    def __init__(self, out_rate=24000):
        self.out_rate = out_rate
        self.audio = bytearray()
        self.lines = []            # 전사 텍스트(시간순)

    def add_audio(self, pcm: bytes, src_rate: int):
        if not pcm:
            return
        if src_rate != self.out_rate:
            a = np.frombuffer(pcm, dtype=np.int16).astype(np.float32)
            n = int(len(a) * self.out_rate / src_rate)
            if n <= 0:
                return
            a = np.interp(
                np.linspace(0, 1, n, endpoint=False),
                np.linspace(0, 1, len(a), endpoint=False), a,
            ).astype(np.int16)
            pcm = a.tobytes()
        self.audio.extend(pcm)

    def add_text(self, who: str, text: str):
        if text and text.strip():
            self.lines.append(f"{who}: {text.strip()}")

    def save(self, base="conversation"):
        if self.audio:
            with wave.open(f"{base}.wav", "wb") as w:
                w.setnchannels(1); w.setsampwidth(2); w.setframerate(self.out_rate)
                w.writeframes(bytes(self.audio))
            secs = len(self.audio) / 2 / self.out_rate
            print(f"💾 대화 합본 저장: {base}.wav ({secs:.1f}s)")
        if self.lines:
            with open(f"{base}.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(self.lines) + "\n")
            print(f"💾 전사 저장: {base}.txt ({len(self.lines)}줄)")

recorder = ConversationRecorder()

# PortAudio 초기화 및 재설정
reset_portaudio()

# Gemini Live 클라이언트 (google-genai)
client = genai.Client(api_key=GEMINI_API_KEY)

# 세션 구성: 음성 출력 + 입/출력 전사 + 한국어 보이스
live_config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    system_instruction=INSTRUCTIONS,
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Charon"),
        ),
        language_code="ko-KR",
    ),
    input_audio_transcription=types.AudioTranscriptionConfig(),   # 사용자 음성 인식
    output_audio_transcription=types.AudioTranscriptionConfig(),  # AI 응답 전사
)

speaker, mic, sender = None, None, None
try:
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    # 마이크 콜백: PCM 데이터를 받아서 asyncio 큐에 넣음
    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    # 마이크: Gemini Live 입력은 16kHz, 모노, 16bit PCM
    mic = sd.RawInputStream(
        samplerate=16000, channels=1, dtype="int16", blocksize=0, callback=mic_cb,
    )

    # Gemini Live 세션 연결
    async with client.aio.live.connect(model=GEMINI_LIVE_MODEL, config=live_config) as session:
        speaker.start()
        mic.start()
        print("🎙️  말해보세요. (중지하려면 커널 인터럽트 ■)\n")
        print("ℹ️  스피커 데모용 에코 차단 모드: AI 가 말하는 동안엔 마이크가 잠시 멈춥니다.\n")

        # 마이크 PCM 을 Gemini 로 전송하는 태스크 (세션 내내 계속 동작)
        # - AI 가 재생 중(또는 직후 가드 구간)이면 전송을 건너뛰어 에코 피드백을 차단
        mic_gate = {"resume_at": 0.0}

        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                now = time.monotonic()
                # 스피커가 아직 재생할 오디오를 갖고 있으면 → 전송 차단 + 가드 갱신
                if speaker.pending_bytes() > 0:
                    mic_gate["resume_at"] = now + MIC_RESUME_GUARD_S
                    continue
                # 재생이 끝나도 잔향이 남는 가드 구간 동안은 계속 차단
                if now < mic_gate["resume_at"]:
                    continue
                recorder.add_audio(chunk, 16000)   # 사용자 음성 녹음(16k→24k 리샘플)
                await session.send_realtime_input(
                    audio=types.Blob(data=chunk, mime_type="audio/pcm;rate=16000"),
                )

        sender = asyncio.create_task(pump_mic())

        # ── 연속 대화 루프 ──────────────────────────────────────────────
        # session.receive() 는 한 턴(turn_complete)이 끝나면 종료되는 제너레이터라,
        # 계속 대화하려면 while 로 감싸 매 턴 다시 호출해야 한다(세션은 그대로 유지).
        while True:
            user_parts, ai_started, turn_um, ai_parts = [], False, None, []   # 매 턴 상태 초기화
            async for msg in session.receive():
                # 응답 오디오(24kHz PCM) → 스피커
                if msg.data:
                    speaker.add(msg.data)
                    recorder.add_audio(msg.data, 24000)   # AI 음성 녹음

                sc = msg.server_content
                if sc:
                    # 사용자 음성 인식(스트리밍) — 모아 두었다가 AI 응답 시작 때 한 줄로 출력
                    if sc.input_transcription and sc.input_transcription.text:
                        user_parts.append(sc.input_transcription.text)
                    # AI 응답 전사(스트리밍)
                    if sc.output_transcription and sc.output_transcription.text:
                        if user_parts and not ai_started:
                            recorder.add_text("🧑 사용자", ''.join(user_parts))
                            print(f"\n🧑 {''.join(user_parts)}")
                            user_parts = []
                        ai_started = True
                        print(sc.output_transcription.text, end="", flush=True)
                        ai_parts.append(sc.output_transcription.text)

                # 이 턴의 usage_metadata 는 갱신되며 도착 → 최신 값만 보관
                if msg.usage_metadata:
                    turn_um = msg.usage_metadata

            # 한 턴 종료(turn_complete) → 이 턴 usage 를 세션 누적에 더하고 출력
            turn = accumulate_turn(turn_um)
            recorder.add_text("🤖 AI", "".join(ai_parts))
            print("\n✅ 응답 완료")
            print_turn_usage(turn)       # 이번 턴
            print_usage_totals()         # 세션 누적
            print()
finally:
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")
    # 세션 종료 시 최종 누적 토큰 사용량 출력
    print("\n=== 세션 누적 토큰 사용량 ===")
    print_usage_totals()
    recorder.save("gemini_speaker")


### (Demo) gemini-3.1-flash-live 끼어들기 O (w/ headset)

헤드셋을 쓰면 에코가 없으므로 마이크 게이팅 없이 **풀듀플렉스 + 진짜 끼어들기(barge-in)** 가 된다. Gemini Live 는 서버 자동 VAD 가 사용자 발화를 감지하면 진행 중 응답을 취소하고 `server_content.interrupted=True` 를 보내는데, 이때 `speaker.clear()` 로 재생을 즉시 끊으면 AI 발화 도중에도 자연스럽게 끼어들 수 있다.

| | 스피커 버전 | 이 헤드셋 버전 |
|---|---|---|
| 마이크 게이팅 | 있음(에코 차단용) | **없음 — 항상 켜짐** |
| 끼어들기 | 사실상 비활성 | **활성 (진짜 barge-in)** |
| `interrupted` 신호 | (게이팅으로 미발생) | **즉시 `speaker.clear()`** |
| 전제 | 에코 환경(스피커) | **헤드셋(에코 없음)** |


In [ ]:
import asyncio
import threading

import sounddevice as sd
from google import genai
from google.genai import types

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 스피커 — 끼어들기 시 clear() 로 재생 즉시 중단 (Gemini Live 출력 24kHz)
class Speaker:
    def __init__(self, samplerate=24000, blocksize=1200):
        self.buf, self.pos = bytearray(), 0
        self.lock = threading.Lock()
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=blocksize, callback=self._cb)

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            avail = len(self.buf) - self.pos
            n = min(need, avail)
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n]); self.pos += n
            if n < need:
                outdata[n:] = b"\x00" * (need - n)
            if self.pos > 2_000_000:
                del self.buf[:self.pos]; self.pos = 0

    def add(self, pcm):
        with self.lock: self.buf.extend(pcm)
    def clear(self):
        with self.lock: del self.buf[:]; self.pos = 0
    def start(self): self.stream.start()
    def close(self):
        try: self.stream.stop()
        finally: self.stream.close()

# ── 토큰 사용량 누적 집계 ───────────────────────────────────────────────
# Gemini 의 usage_metadata 는 "이번 턴(1회 추론)"의 사용량(per-request)이다.
# (대화 이력이 매 턴 입력 프롬프트에 다시 포함되므로 input 값이 턴마다 커 보이지만,
#  그건 누적이 아니라 그 턴의 입력 크기다.) 세션 전체 비용을 보려면 gpt-realtime
# 처럼 턴마다 값을 "더해야" 한다.
usage_totals = {
    "input": 0, "input_text": 0, "input_audio": 0, "input_cached": 0,
    "output": 0, "output_text": 0, "output_audio": 0, "total": 0,
}

def _modality(d):
    return str(getattr(d.modality, "name", d.modality)).upper()

def _extract_usage(u):
    # 이번 턴의 usage_metadata 를 dict 로 정규화 (per-turn 스냅샷)
    snap = dict.fromkeys(usage_totals, 0)
    if u is None:
        return snap
    snap["input"]        = u.prompt_token_count or 0
    snap["output"]       = u.response_token_count or 0
    snap["total"]        = u.total_token_count or 0
    snap["input_cached"] = getattr(u, "cached_content_token_count", 0) or 0
    for d in (u.prompt_tokens_details or []):
        if "AUDIO" in _modality(d): snap["input_audio"] = d.token_count or 0
        elif "TEXT" in _modality(d): snap["input_text"] = d.token_count or 0
    for d in (u.response_tokens_details or []):
        if "AUDIO" in _modality(d): snap["output_audio"] = d.token_count or 0
        elif "TEXT" in _modality(d): snap["output_text"] = d.token_count or 0
    return snap

def accumulate_turn(u):
    # 한 턴이 끝날 때 그 턴의 usage 를 세션 누적치에 더하고 그 턴 값을 돌려준다
    snap = _extract_usage(u)
    for k in usage_totals:
        usage_totals[k] += snap[k]
    return snap

def print_turn_usage(turn):
    print(
        f"🔹 이번 턴: 총 {turn['total']:,} | "
        f"입력 {turn['input']:,} "
        f"(텍스트 {turn['input_text']:,} / 오디오 {turn['input_audio']:,} / 캐시 {turn['input_cached']:,}) | "
        f"출력 {turn['output']:,} "
        f"(텍스트 {turn['output_text']:,} / 오디오 {turn['output_audio']:,})"
    )

def print_usage_totals():
    print(
        f"📊 누적 토큰: 총 {usage_totals['total']:,} | "
        f"입력 {usage_totals['input']:,} "
        f"(텍스트 {usage_totals['input_text']:,} / 오디오 {usage_totals['input_audio']:,} / 캐시 {usage_totals['input_cached']:,}) | "
        f"출력 {usage_totals['output']:,} "
        f"(텍스트 {usage_totals['output_text']:,} / 오디오 {usage_totals['output_audio']:,})"
    )

# ── 대화 녹음(합본 WAV) + 전사 텍스트 저장 ─────────────────────────────
# 사용자·AI 오디오를 하나의 WAV 로 합본한다. 반이중/턴테이킹 구조라 도착 순서대로
# 이어붙이면 대화 타임라인이 되고, 샘플레이트가 다르면 out_rate 로 선형 리샘플한다.
import wave
import numpy as np

class ConversationRecorder:
    def __init__(self, out_rate=24000):
        self.out_rate = out_rate
        self.audio = bytearray()
        self.lines = []            # 전사 텍스트(시간순)

    def add_audio(self, pcm: bytes, src_rate: int):
        if not pcm:
            return
        if src_rate != self.out_rate:
            a = np.frombuffer(pcm, dtype=np.int16).astype(np.float32)
            n = int(len(a) * self.out_rate / src_rate)
            if n <= 0:
                return
            a = np.interp(
                np.linspace(0, 1, n, endpoint=False),
                np.linspace(0, 1, len(a), endpoint=False), a,
            ).astype(np.int16)
            pcm = a.tobytes()
        self.audio.extend(pcm)

    def add_text(self, who: str, text: str):
        if text and text.strip():
            self.lines.append(f"{who}: {text.strip()}")

    def save(self, base="conversation"):
        if self.audio:
            with wave.open(f"{base}.wav", "wb") as w:
                w.setnchannels(1); w.setsampwidth(2); w.setframerate(self.out_rate)
                w.writeframes(bytes(self.audio))
            secs = len(self.audio) / 2 / self.out_rate
            print(f"💾 대화 합본 저장: {base}.wav ({secs:.1f}s)")
        if self.lines:
            with open(f"{base}.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(self.lines) + "\n")
            print(f"💾 전사 저장: {base}.txt ({len(self.lines)}줄)")

recorder = ConversationRecorder()

reset_portaudio()

# Gemini Live 클라이언트 (google-genai)
client = genai.Client(api_key=GEMINI_API_KEY)

live_config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    system_instruction=INSTRUCTIONS,
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Charon"),
        ),
        language_code="ko-KR",
    ),
    input_audio_transcription=types.AudioTranscriptionConfig(),
    output_audio_transcription=types.AudioTranscriptionConfig(),
)

speaker, mic, sender = None, None, None
try:
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    # 마이크: Gemini Live 입력은 16kHz (100ms 블록 = 1600 프레임)
    mic = sd.RawInputStream(
        samplerate=16000, channels=1, dtype="int16",
        blocksize=1600, latency="high", callback=mic_cb,
    )

    async with client.aio.live.connect(model=GEMINI_LIVE_MODEL, config=live_config) as session:
        speaker.start()
        mic.start()
        print("🎧 헤드셋 착용 후 말해보세요. AI 발화 도중에 끼어들 수 있어요. (중지: 커널 인터럽트 ■)\n")

        # 헤드셋 → 에코 없음 → 마이크를 항상 켜둔다(게이팅 X = 진짜 full-duplex)
        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                recorder.add_audio(chunk, 16000)   # 사용자 음성 녹음(16k→24k 리샘플)
                await session.send_realtime_input(
                    audio=types.Blob(data=chunk, mime_type="audio/pcm;rate=16000"),
                )

        sender = asyncio.create_task(pump_mic())

        # ── 연속 대화 루프 ──────────────────────────────────────────────
        # session.receive() 는 한 턴(turn_complete)이 끝나면 종료되는 제너레이터라,
        # 계속 대화하려면 while 로 감싸 매 턴 다시 호출해야 한다(세션은 그대로 유지).
        while True:
            user_parts, ai_started, turn_um, ai_parts = [], False, None, []   # 매 턴 상태 초기화
            async for msg in session.receive():
                if msg.data:
                    speaker.add(msg.data)
                    recorder.add_audio(msg.data, 24000)   # AI 음성 녹음
                if msg.usage_metadata:
                    turn_um = msg.usage_metadata   # 이 턴의 최신 usage 보관

                sc = msg.server_content
                if sc:
                    if sc.input_transcription and sc.input_transcription.text:
                        user_parts.append(sc.input_transcription.text)
                    if sc.output_transcription and sc.output_transcription.text:
                        if user_parts and not ai_started:
                            recorder.add_text("🧑 사용자", ''.join(user_parts))
                            print(f"\n🧑 {''.join(user_parts)}")
                            user_parts = []
                        ai_started = True
                        print(sc.output_transcription.text, end="", flush=True)
                        ai_parts.append(sc.output_transcription.text)
                    # 끼어들기: 서버가 사용자 발화로 현재 응답을 취소 → 재생 즉시 중단
                    if sc.interrupted:
                        print("\n🎧 (끼어들기) 음성 인식 시작...")
                        speaker.clear()
                        ai_started = False

            # 한 턴 종료 → 이 턴 usage 를 세션 누적에 더하고 출력
            turn = accumulate_turn(turn_um)
            recorder.add_text("🤖 AI", "".join(ai_parts))
            print("\n✅ 응답 완료")
            print_turn_usage(turn)       # 이번 턴
            print_usage_totals()         # 세션 누적
            print()
finally:
    if sender:
        sender.cancel()
    if mic:
        try: mic.stop(); mic.close()
        except Exception as e: print(f"마이크 종료 에러: {e}")
    if speaker:
        try: speaker.close()
        except Exception as e: print(f"스피커 종료 에러: {e}")
    print("\n=== 세션 누적 토큰 사용량 ===")
    print_usage_totals()
    recorder.save("gemini_headset")
